# Fraud Detection

You're an ML engineer at a retail bank. Millions of transactions flow through the system every day, and your job is to flag potentially fraudulent ones for the investigations team.

The vast majority of transactions are legitimate. Fraud is rare but costly. A **false positive** means blocking a real customer's card and triggering a needless investigation. A **false negative** means the fraud goes undetected.

In this lab you'll see why accuracy is a trap for imbalanced datasets, learn about **precision** and the **precision-recall trade-off**, and explore strategies for handling class imbalance.

## Loading and exploring the data

The first thing we need to understand about this dataset is just how rare fraud is. In most real-world transaction data, fraudulent transactions make up well under 1% of the total. This extreme imbalance is what makes fraud detection a very different challenge from the cancer detection you did in the previous unit.

Let's load the data and see what we're dealing with.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             precision_recall_curve, roc_curve, roc_auc_score)

In [ ]:
df = pd.read_csv("./transactions.csv")
print(f"Shape: {df.shape}")
print(f"\nClass distribution:")
print(df["is_fraud"].value_counts())
print(f"\nFraud rate: {df['is_fraud'].mean():.1%}")

That fraud rate is tiny. Let's visualise the imbalance to really see the scale of the problem. Compare this to the breast cancer task, where the split was roughly 63/37.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
df["is_fraud"].value_counts().plot(kind="bar", ax=ax, color=["steelblue", "coral"])
ax.set_xticklabels(["Legitimate", "Fraud"], rotation=0)
ax.set_ylabel("Count")
ax.set_title("Transaction class distribution")
plt.tight_layout()
plt.show()

## The accuracy trap

What happens if we just train a standard logistic regression on this data without thinking about the imbalance? Let's find out. We'll do the usual split, scale, train, and predict, then look at the results.

In [ ]:
X = df.drop(columns=["is_fraud"])
y = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train with default settings, no class weighting
model_default = LogisticRegression(max_iter=1000, random_state=42)
model_default.fit(X_train_scaled, y_train)
y_pred_default = model_default.predict(X_test_scaled)

print(f"Accuracy: {accuracy_score(y_test, y_pred_default):.1%}")
print(f"Fraud cases in test set: {y_test.sum()}")
print(f"Fraud cases caught: {((y_pred_default == 1) & (y_test == 1)).sum()}")
print(f"Legitimate flagged as fraud: {((y_pred_default == 1) & (y_test == 0)).sum()}")

The model gets ~98% accuracy, which sounds good until you look at the numbers. It catches almost no fraud. A model that predicts "legitimate" for every single transaction would get the same accuracy. When one class dominates, accuracy is meaningless.

## Introducing precision

In the previous unit you learned recall: of all the actual positives, how many did we catch? Now meet precision: **of all the transactions we flagged as fraud, how many actually were?**

Low precision means the investigations team is drowning in false alarms. Real customers having their cards blocked, investigator time wasted.

In [ ]:
precision = precision_score(y_test, y_pred_default)
recall = recall_score(y_test, y_pred_default)
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")

## The precision-recall tension

You learned recall in the cancer detection unit: don't miss a cancer. Now you've learned precision: don't cry wolf. The problem is that these two metrics pull in opposite directions.

- Flag more transactions: catch more fraud (recall goes up), but more false alarms (precision goes down)
- Flag fewer transactions: fewer false alarms (precision goes up), but miss more fraud (recall goes down)

There is no free lunch. Every system that flags rare events must choose where to sit on this trade-off.

## Strategies for imbalanced data

So the default model doesn't work well for fraud detection. The good news is there are practical strategies to help the model pay more attention to the minority class. We'll try two of them and compare the results.

### Strategy 1: Class weighting

The simplest approach is to tell the model that mistakes on fraud cases should count for more during training. Setting `class_weight="balanced"` automatically scales the penalty in proportion to how rare each class is, so missing a fraud case hurts the model much more than misclassifying a legitimate one.

In [ ]:
# Train a new logistic regression with class_weight="balanced"
model_balanced = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
model_balanced.fit(X_train_scaled, y_train)
y_pred_balanced = model_balanced.predict(X_test_scaled)

print(f"Precision: {precision_score(y_test, y_pred_balanced):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred_balanced):.3f}")

`class_weight="balanced"` tells the model to penalise errors on the minority class more heavily. Recall has jumped up, meaning the model now catches more fraud, but look at precision: it's dropped. The model is casting a wider net, which means more legitimate transactions are getting flagged too. This is the precision-recall tension in action.

### Strategy 2: Oversampling

Instead of changing how the model weighs errors, we can change the training data itself. The idea is simple: duplicate the fraud transactions until they roughly match the number of legitimate ones. This gives the model more fraud examples to learn from, without needing to change the algorithm.

In [ ]:
# Oversample the minority class by duplicating fraud transactions
fraud_train = X_train_scaled[y_train == 1]
fraud_labels = y_train[y_train == 1]

# Duplicate fraud samples until they roughly match the majority class
repeat_factor = (y_train == 0).sum() // (y_train == 1).sum()
X_oversampled = np.vstack([X_train_scaled, np.repeat(fraud_train, repeat_factor, axis=0)])
y_oversampled = np.concatenate([y_train, np.repeat(fraud_labels, repeat_factor)])

model_oversampled = LogisticRegression(max_iter=1000, random_state=42)
model_oversampled.fit(X_oversampled, y_oversampled)
y_pred_over = model_oversampled.predict(X_test_scaled)

print(f"Precision: {precision_score(y_test, y_pred_over):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred_over):.3f}")

### Comparison

Now let's put all three approaches side by side so we can see the trade-offs clearly. Notice how each strategy shifts the balance between precision and recall differently.

In [ ]:
# Comparison table showing precision and recall for each strategy
comparison = pd.DataFrame({
    "Strategy": ["Default", "Class weighted", "Oversampled"],
    "Precision": [
        precision_score(y_test, y_pred_default),
        precision_score(y_test, y_pred_balanced),
        precision_score(y_test, y_pred_over),
    ],
    "Recall": [
        recall_score(y_test, y_pred_default),
        recall_score(y_test, y_pred_balanced),
        recall_score(y_test, y_pred_over),
    ],
})
comparison.style.format({"Precision": "{:.3f}", "Recall": "{:.3f}"})

## Decision thresholds revisited

In the cancer detection lab, you saw how lowering the decision threshold increases recall. The same idea applies here, but now we can visualise the full trade-off between precision and recall across every possible threshold. This is the **precision-recall curve**.

In [ ]:
y_proba = model_balanced.predict_proba(X_test_scaled)[:, 1]
precisions, recalls, thresholds_pr = precision_recall_curve(y_test, y_proba)

plt.figure(figsize=(8, 5))
plt.plot(recalls, precisions)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Every point on this curve represents a different decision threshold. Moving right increases recall (catch more fraud) but decreases precision (more false alarms). The shape of the curve tells you how severe the trade-off is.

## ROC curve

The precision-recall curve shows us the trade-off from the model's perspective. The **ROC curve** shows a related but different view: it plots the true positive rate (recall) against the false positive rate at every threshold. It also gives us a single summary number, **AUC** (area under the curve), that's useful for comparing models at a glance.

In [ ]:
fpr, tpr, thresholds_roc = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

plt.figure(figsize=(8, 5))
plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", color="grey", label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("ROC Curve")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

The ROC curve plots true positive rate (recall) against false positive rate. AUC (area under the curve) summarises overall performance in a single number: 0.5 = random, 1.0 = perfect. It's useful for comparing models, but doesn't tell you which operating point to choose.

## Choosing an operating point

The bank's fraud team says: "We need to catch at least 80% of fraud (recall ≥ 0.8), but we can only investigate 50 flagged transactions per day."

In [ ]:
# Find the threshold that gives recall >= 0.8
for i, (p, r) in enumerate(zip(precisions, recalls)):
    if r >= 0.8 and r < 0.85:
        print(f"Threshold ≈ {thresholds_pr[i]:.3f}: Precision = {p:.3f}, Recall = {r:.3f}")

**Think about these questions:**

1. At this operating point, how many false alarms would the team see per day (assuming 10,000 transactions/day)?
2. Is this acceptable? What if the team could only investigate 20 per day?
3. How would you present this trade-off to the bank's head of fraud?